# Store Revenue Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `revenue`

## 2 · Project Overview

We predict **monthly store revenue** based on store size, employee count, location type, category, foot traffic, average transaction value, parking, competition, and store maturity.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a retail store's physical attributes, location, traffic, and competitive environment, predict its monthly revenue.

## 5 · Why This Project Matters

- **Store revenue forecasting** drives inventory, staffing, and expansion decisions.
- Understanding revenue drivers helps optimize location selection.
- Competition effects are directly measurable.
- Demonstrates regression with mixed feature types and business interactions.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | store_size_sqft, num_employees, location_type, store_category, foot_traffic_daily, avg_transaction_value, parking_spaces, competitor_count, months_open |
| **Target** | revenue (continuous, USD) |
| **Range** | ~$5K – $500K |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "revenue"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 retail store monthly revenue records with location and store attributes.

In [ ]:
np.random.seed(SEED)
n = 8000
store_size_sqft = np.random.choice([1500, 3000, 5000, 8000, 12000], n,
                                    p=[0.15, 0.25, 0.3, 0.2, 0.1])
num_employees = (store_size_sqft / 500 + np.random.normal(0, 1, n)).clip(2, 30).astype(int)
location_type = np.random.choice(["Mall", "Downtown", "Suburban", "Highway"], n,
                                  p=[0.25, 0.3, 0.25, 0.2])
loc_mult = {"Mall": 1.3, "Downtown": 1.15, "Suburban": 1.0, "Highway": 0.85}
loc_val = np.array([loc_mult[l] for l in location_type])

store_category = np.random.choice(["Grocery", "Electronics", "Clothing", "Home", "Pharmacy"], n,
                                   p=[0.25, 0.15, 0.2, 0.2, 0.2])
cat_base = {"Grocery": 40000, "Electronics": 60000, "Clothing": 35000, "Home": 45000, "Pharmacy": 50000}
cat_val = np.array([cat_base[c] for c in store_category])

foot_traffic_daily = np.round(np.random.lognormal(5.5, 0.6, n).clip(50, 5000), 0).astype(int)
avg_transaction_value = np.round(np.random.normal(35, 12, n).clip(8, 120), 2)
parking_spaces = (store_size_sqft / 200 + np.random.normal(0, 3, n)).clip(0, 80).astype(int)
competitor_count = np.random.poisson(3, n).clip(0, 12)
months_open = np.random.randint(1, 240, n)

# Revenue model
revenue = (cat_val
           + 0.8 * foot_traffic_daily * avg_transaction_value
           + 2.5 * store_size_sqft
           + 500 * num_employees
           - 3000 * competitor_count
           + 50 * parking_spaces
           + 20 * np.minimum(months_open, 60))
revenue = revenue * loc_val
revenue = np.round(revenue + np.random.normal(0, 8000, n), 2).clip(5000, 500000)

df = pd.DataFrame({
    "store_size_sqft": store_size_sqft, "num_employees": num_employees,
    "location_type": location_type, "store_category": store_category,
    "foot_traffic_daily": foot_traffic_daily, "avg_transaction_value": avg_transaction_value,
    "parking_spaces": parking_spaces, "competitor_count": competitor_count,
    "months_open": months_open, "revenue": revenue
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['revenue'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["foot_traffic_daily"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Daily Foot Traffic"); axes[0][0].set_ylabel("Revenue")
axes[0][0].set_title("Foot Traffic vs Revenue")

axes[0][1].scatter(df["store_size_sqft"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Store Size (sqft)"); axes[0][1].set_ylabel("Revenue")
axes[0][1].set_title("Store Size vs Revenue")

axes[0][2].scatter(df["avg_transaction_value"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Avg Transaction ($)"); axes[0][2].set_ylabel("Revenue")
axes[0][2].set_title("Transaction Value vs Revenue")

df.boxplot(column=TARGET, by="location_type", ax=axes[1][0])
axes[1][0].set_title("Revenue by Location Type")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="store_category", ax=axes[1][1])
axes[1][1].set_title("Revenue by Category")
axes[1][1].tick_params(axis="x", rotation=45)

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `revenue`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Revenue ($)")

df.boxplot(column=TARGET, by="location_type", ax=axes[1])
axes[1].set_title("Revenue by Location Type")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `location_type`, `store_category` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: High-revenue stores are genuine — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["revenue_per_sqft_proxy"] = X_train["foot_traffic_daily"] * X_train["avg_transaction_value"] / (X_train["store_size_sqft"] + 1)
X_test["revenue_per_sqft_proxy"] = X_test["foot_traffic_daily"] * X_test["avg_transaction_value"] / (X_test["store_size_sqft"] + 1)

X_train["traffic_x_transaction"] = X_train["foot_traffic_daily"] * X_train["avg_transaction_value"]
X_test["traffic_x_transaction"] = X_test["foot_traffic_daily"] * X_test["avg_transaction_value"]

X_train["maturity"] = np.minimum(X_train["months_open"], 60)
X_test["maturity"] = np.minimum(X_test["months_open"], 60)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Foot traffic × transaction value** is the strongest revenue driver.
- **Store size** contributes through capacity and display area.
- **Mall locations** generate ~30% more revenue than suburban.
- **Competitor count** has a clear negative impact (~$3K per competitor).
- **Store maturity** plateaus around 5 years.

**Business takeaway:** Maximize foot traffic (marketing, location) and average transaction value (upselling, product mix) for revenue growth.

## 26 · Limitations

1. Synthetic data with simplified revenue dynamics.
2. No seasonal or promotional effects.
3. Missing online/omnichannel revenue.
4. No customer demographic data.
5. Simplified competition model.

## 27 · How to Improve This Project

1. Use real POS transaction data.
2. Add seasonal and promotional features.
3. Include online sales channel.
4. Add customer demographics and loyalty data.
5. Model time-series revenue trends.

## 28 · Production Considerations

- Deploy for store revenue forecasting dashboards.
- Use for new store location evaluation.
- Monitor for revenue anomalies.
- Update monthly with actual sales data.
- Output prediction intervals for budgeting.

## 29 · Common Mistakes

1. Ignoring the interaction between traffic and transaction value.
2. Not capping store maturity (diminishing returns).
3. Treating all store categories equally.
4. Not accounting for local competition.
5. Using raw months_open without diminishing returns.

## 30 · Mini Challenge / Exercises

1. Create `revenue_per_employee` and analyze efficiency.
2. Remove `competitor_count` — how much does RMSE increase?
3. Plot revenue by store maturity — confirm the plateau.
4. Build separate models by store_category.
5. Add `traffic_per_sqft` as a density metric.

## 31 · Final Summary / Key Takeaways

1. **Foot traffic × transaction value** is the primary revenue driver.
2. **Store size** contributes through capacity effects.
3. **Location type** (Mall > Downtown > Suburban > Highway) sets the revenue tier.
4. **Competition** has a measurable negative impact.
5. **Store maturity** plateaus around 60 months.
6. **Interaction features** significantly improve predictions.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))